# Imports

In [72]:
import os
import time
import gc
import sys
import numpy as np
import pandas as pd
import psutil
import requests
from PIL import Image
from tqdm.notebook import tqdm

In [73]:
import cv2

In [74]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from transformers import (
    ViTModel, ViTImageProcessor,
    AutoImageProcessor, AutoModel,
    DeiTModel, DeiTImageProcessor,
    CLIPProcessor, CLIPVisionModel, CLIPTextModel,
    BertTokenizer, BertModel, 
    RobertaTokenizer, RobertaModel,
    GPT2Tokenizer, GPT2Model
)

In [75]:
import torch
from torchvision import models

# Configuration

In [76]:
#("Flickr8k", "Flickr30k", "ConceptualCaptions")
CURRENT_DATASET = "Flickr8k" 
ALL_DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]

# --- DIRECTORY ARCHITECTURE ---
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')
DATASETS_DIR = os.path.join(DATA_DIR, 'Datasets')
RAW_DIR = os.path.join(DATA_DIR, 'Features_RAW')
RESULTS_DIR = os.path.join(DATA_DIR, 'Results')

# Create core directories
for d in [DATASETS_DIR, RAW_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Function to create dataset-specific saliency folders
def get_saliency_dir(dataset_name):
    path = os.path.join(DATA_DIR, f"SaliencyMaps_{dataset_name}")
    os.makedirs(path, exist_ok=True)
    return path

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Environment initialized. Using device: {device}")

Environment initialized. Using device: cuda


# Data Loading

In [77]:
print(f"Loading dataset: {CURRENT_DATASET}...")

df_path = os.path.join(DATASETS_DIR, f"df_{CURRENT_DATASET}.pkl")
if not os.path.exists(df_path):
    raise FileNotFoundError(f"Dataset file not found at {df_path}. Please run data_builder.ipynb first.")

df = pd.read_pickle(df_path)

# Extract inputs for models
IMAGE_PATHS = df['image_path'].tolist()
ALIGNED_CAPTIONS = [caps[0] for caps in df['captions']]

print(f"Ready. Loaded {len(IMAGE_PATHS)} images and {len(ALIGNED_CAPTIONS)} captions into memory.")

Loading dataset: Flickr8k...
Ready. Loaded 8091 images and 8091 captions into memory.


# Utility Functions
## GreenAI Metrics

In [78]:
def send_ntfy_notification(message, title="TFE Pipeline Update"):
    """Sends a push notification via ntfy.sh."""
    try:
        requests.post(
            f"https://ntfy.sh/{'aysel_tfe_server_9988'}",
            data=message.encode(encoding='utf-8'),
            headers={"Title": title}
        )
    except Exception as e:
        print(f"Notification failed: {e}")

def measure_memory():
    """Returns the current memory usage of the process in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

def get_size_in_mb(obj):
    """Returns the size of an object in MB."""
    return sys.getsizeof(obj) / (1024 * 1024)



In [79]:
def execute_and_save(modality, model_name, extract_func, data, device):
    """Measures metrics, extracts features + saliency, and saves results."""
    print(f"\nProcessing {modality.upper()} with model: {model_name}")
    start_time = time.time()
    mem_before = measure_memory()
    
    features = extract_func(data, device)
    
    exec_time = time.time() - start_time
    mem_used = max(0, measure_memory() - mem_before)
    
    save_path = os.path.join(RAW_DIR, f"X_{modality}_{model_name}_raw_{CURRENT_DATASET}.npy")
    np.save(save_path, features)
    
    print(f"Saved RAW features to: {save_path}")
    original_dim = features.shape[1]
    original_size = get_size_in_mb(features)
    
    del features
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
        
    return {
        "Modality": modality, "Model": model_name, "Original_Dim": original_dim,
        "Exec_Time_s": exec_time, "Memory_Used_MB": mem_used, "Original_Size_MB": original_size
    }

## XAI Saliency Maps

In [80]:
def save_saliency_and_overlay(model_name, activations, img_idx, img_path, dataset):
    """
    Captures pertinent zones using Norm-based weighting.
    Fixes the token dimension mismatch by explicitly removing the [CLS] token.
    """
    act = activations.detach().cpu()
    
    # --- CNN Logic (4D Tensors) ---
    if len(act.shape) == 4:
        weights = torch.norm(act, dim=1, keepdim=True)
        heatmap = torch.sum(act * weights, dim=1).squeeze().numpy()
        
    # --- Transformer Logic (3D Tensors) ---
    elif len(act.shape) == 3:
        # act shape: [1, 197, 768] (for DeiT/ViT)
        # We must remove the [CLS] token at index 0
        tokens = act[:, 1:, :] # Shape: [1, 196, 768]
        
        # Calculate importance weights for the 196 patches
        weights = torch.norm(tokens, dim=-1, keepdim=True) # Shape: [1, 196, 1]
        
        # Apply weighting and sum feature dimensions -> Shape: [1, 196]
        weighted_sum = torch.sum(tokens * weights, dim=-1)
        
        # Calculate grid size (sqrt of 196 is 14)
        num_patches = tokens.shape[1]
        side = int(np.sqrt(num_patches)) 
        
        try:
            # Reshape exactly 196 elements into [14, 14]
            heatmap = weighted_sum.view(side, side).numpy()
        except RuntimeError:
            # Fallback if patch count is unexpected
            return
            
    else: return

    # Normalize heatmap 0-1
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) > 0: heatmap /= np.max(heatmap)
    heatmap_res = cv2.resize(heatmap, (224, 224))
    
    # Save Raw Saliency Map (.npy)
    saliency_dir = os.path.join(DATA_DIR, f"SaliencyMaps_{dataset}")
    os.makedirs(saliency_dir, exist_ok=True)
    np.save(os.path.join(saliency_dir, f"Saliency_{model_name}_{img_idx}.npy"), heatmap_res)
    
    # Save Visual Overlay 
    img_bgr = cv2.imread(img_path)
    img_bgr = cv2.resize(img_bgr, (224, 224))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_res), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img_bgr, 0.6, heatmap_color, 0.4, 0)
    cv2.imwrite(os.path.join(RESULTS_DIR, f"Overlay_{dataset}_{model_name}_{img_idx}.jpg"), overlay)

# Unimodal Models
## Indexing: Embedding Models

## Vision Feature Extractions

In [109]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            # 1. Force RGB at the PIL level
            image = Image.open(img_path).convert('RGB')
            
            # 2. Additional safety: check if PIL actually produced 3 channels
            if len(image.getbands()) != 3:
                image = image.convert('RGB')
                
        except Exception:
            # Placeholder for corrupted web-scraped images
            image = Image.new('RGB', (224, 224), (0, 0, 0)) 
            
        if self.transform:
            try:
                image = self.transform(image)
            except Exception:
                # If the transform still fails (rare), return a zero tensor
                return torch.zeros((3, 224, 224))
        return image

def extract_vision_features(model, model_name, dataloader, device, target_layer, paths, dataset_name):
    model.eval()
    features, activations, img_idx = [], {}, 0

    def hook_fn(module, input, output):
        # Handle the specific tuple/object output of Transformer Layers
        if isinstance(output, tuple):
            activations['value'] = output[0].detach()
        elif hasattr(output, 'last_hidden_state'):
            activations['value'] = output.last_hidden_state.detach()
        else:
            activations['value'] = output.detach()

    # Register hook
    hook = target_layer.register_forward_hook(hook_fn)
    safe_name = model_name.lower()

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Extraction: {safe_name}"):
            batch = batch.to(device)
            
            # For DeiT/ViT, ensure we aren't masking anything
            outputs = model(batch)
            
            # Global Feature (CLS token)
            if hasattr(outputs, 'last_hidden_state'):
                feat = outputs.last_hidden_state[:, 0, :]
            elif isinstance(outputs, torch.Tensor):
                feat = outputs[:, 0, :]
            else:
                # Access first element of BaseModelOutput
                feat = outputs[0][:, 0, :]
                
            features.append(feat.cpu().numpy())

            # --- SALIENCY GENERATION ---
            batch_acts = activations.get('value')
            if batch_acts is not None:
                # Correct number of tokens check (should be 197 for DeiT)
                for i in range(batch_acts.size(0)):
                    save_saliency_and_overlay(safe_name, batch_acts[i:i+1], img_idx, paths[img_idx], dataset_name)
                    img_idx += 1
                activations['value'] = None 
            else:
                # If this prints, the hook is the problem
                print(f"DEBUG: Hook failed to capture for {safe_name} at index {img_idx}")

    hook.remove()
    return np.vstack(features)

def get_resnet50_embeddings(paths, device, dataset):
    m = models.resnet50(weights="DEFAULT").to(device)
    t = m.layer4[-1] 
    m.fc = nn.Identity()
    loader = torch.utils.data.DataLoader(ImageDataset(paths, models.ResNet50_Weights.DEFAULT.transforms()), batch_size=32)
    return extract_vision_features(m, "resnet50", loader, device, t, paths, dataset)

def get_mobilenet_v3_embeddings(paths, device, dataset):
    m = models.mobilenet_v3_large(weights="DEFAULT").to(device)
    t = m.features[-1]
    m.classifier = nn.Identity()
    loader = torch.utils.data.DataLoader(ImageDataset(paths, models.MobileNet_V3_Large_Weights.DEFAULT.transforms()), batch_size=32)
    return extract_vision_features(m, "mobilenet_v3", loader, device, t, paths, dataset)

def get_vit_embeddings(paths, device, dataset):
    proc = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
    m = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device)
    t = m.encoder.layer[-1]
    tf = lambda x: proc(images=x, return_tensors="pt")['pixel_values'].squeeze(0)
    loader = torch.utils.data.DataLoader(ImageDataset(paths, tf), batch_size=32)
    return extract_vision_features(m, "vit", loader, device, t, paths, dataset)


def get_deit_embeddings(image_paths, device, dataset_name):
    # Use explicit classes to avoid AutoModel ambiguity
    processor = DeiTImageProcessor.from_pretrained('facebook/deit-base-distilled-patch16-224')
    model = DeiTModel.from_pretrained('facebook/deit-base-distilled-patch16-224').to(device)

    if hasattr(model, 'deit'):
        target_layer = model.deit.encoder.layer[-1]
    else:
        target_layer = model.encoder.layer[-1]
    
    print(f"Targeting Layer: {target_layer.__class__.__name__}")

    def transform(img): 
        return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    
    ds_obj = ImageDataset(image_paths, transform=transform)
    dataloader = torch.utils.data.DataLoader(ds_obj, batch_size=32, num_workers=4)
    
    return extract_vision_features(model, "deit", dataloader, device, target_layer, image_paths, dataset_name)

def get_clip_vision_embeddings(image_paths, device, dataset_name):
    processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    model = CLIPVisionModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
    
    def transform(img): 
        return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    
    ds = ImageDataset(image_paths, transform=transform)
    dataloader = torch.utils.data.DataLoader(ds, batch_size=32, num_workers=4)
    target_layer = model.vision_model.encoder.layers[-1]
    
    return extract_vision_features(
        model, 
        "clip_vision", 
        dataloader, 
        device, 
        target_layer, 
        image_paths, 
        dataset_name
    )

## T2T Text 


### Indexing: Embedding Models

In [82]:
## Text Feature Extraction
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        return {key: val.squeeze(0) for key, val in encoding.items()}

def extract_text_features(model, dataloader, device, feature_type='last_hidden_state_mean'):
    model.eval()
    features = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting Text"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            
            if feature_type == 'last_hidden_state_mean':
                attention_mask = batch['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
                sum_embeddings = torch.sum(outputs.last_hidden_state * attention_mask, 1)
                sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
                batch_features = (sum_embeddings / sum_mask).cpu().numpy()
            else:
                batch_features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            features.append(batch_features)
    return np.vstack(features)

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        output, (hidden, cell) = self.rnn(embedded)
        return hidden[-1]

def get_rnn_embeddings(texts, device):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = SimpleRNN(tokenizer.vocab_size).to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    
    model.eval()
    features = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting RNN"):
            output = model(batch['input_ids'].to(device))
            features.append(output.cpu().numpy())
    return np.vstack(features)

def get_bert_embeddings(texts, device):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return extract_text_features(model, dataloader, device)

def get_roberta_embeddings(texts, device):
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    model = RobertaModel.from_pretrained('roberta-base').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return extract_text_features(model, dataloader, device)

def get_gpt2_embeddings(texts, device):
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2Model.from_pretrained('gpt2').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return extract_text_features(model, dataloader, device)

def get_clip_text_embeddings(texts, device):
    from transformers import CLIPModel, CLIPProcessor
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    features = []
    model.eval()
    with torch.no_grad():
        batch_size = 32
        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting CLIP Text"):
            batch_texts = texts[i:i+batch_size]
            inputs = processor(text=batch_texts, return_tensors="pt", padding=True, truncation=True).to(device)
            outputs = model.get_text_features(**inputs)
            
            if hasattr(outputs, 'text_embeds'):
                batch_features = outputs.text_embeds.cpu().numpy()
            elif hasattr(outputs, 'pooler_output'):
                batch_features = outputs.pooler_output.cpu().numpy()
            else:
                batch_features = outputs.cpu().numpy()
            features.append(batch_features)
    return np.vstack(features)

## Execution

In [83]:
vision_pipeline = {
    "resnet50": get_resnet50_embeddings,
    "mobilenet_v3": get_mobilenet_v3_embeddings,
    "vit": get_vit_embeddings,
    "deit": get_deit_embeddings,
    "clip_vision": get_clip_vision_embeddings
}

text_pipeline = {
    "rnn": get_rnn_embeddings,
    "bert": get_bert_embeddings,
    "roberta": get_roberta_embeddings,
    "gpt2": get_gpt2_embeddings,
    "clip_text": get_clip_text_embeddings
}

ALL_DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]
# --- Global Execution Loop ---
all_metrics = []

for ds_name in ALL_DATASETS:
    # 1. Dataset Loading
    df_path = os.path.join(DATASETS_DIR, f"df_{ds_name}.pkl")
    if not os.path.exists(df_path):
        print(f"Skipping {ds_name}: Metadata not found."); continue
        
    df = pd.read_pickle(df_path)
    IMAGE_PATHS = df['image_path'].tolist()
    ALIGNED_CAPTIONS = [c[0] if isinstance(c, list) else c for c in df['captions']]

    send_ntfy_notification(f"Starting Indexation for {ds_name}", "TFE Pipeline")

    # 2. Vision Pipeline Execution
    print(f"\n{'='*20}\nVISION: {ds_name}\n{'='*20}")
    # Note: We use a lambda to pass the dataset name into the wrapper
    vision_pipeline = {
        "resnet50": lambda p, d: get_resnet50_embeddings(p, d, ds_name),
        "mobilenet_v3": lambda p, d: get_mobilenet_v3_embeddings(p, d, ds_name),
        "vit": lambda p, d: get_vit_embeddings(p, d, ds_name),
        "deit": lambda p, d: get_deit_embeddings(p, d, ds_name),
        "clip_vision": lambda p, d: get_clip_vision_embeddings(p, d, ds_name)
    }

    for name, func in vision_pipeline.items():
        metrics = execute_and_save("vision", name, func, IMAGE_PATHS, device)
        all_metrics.append({**metrics, "Dataset": ds_name})
    
    # 3. Text Pipeline Execution
    print(f"\n{'='*20}\nTEXT: {ds_name}\n{'='*20}")
    # Text models do not require the 'ds_name' inside the wrapper
    text_pipeline = {
        "rnn": get_rnn_embeddings, "bert": get_bert_embeddings,
        "roberta": get_roberta_embeddings, "gpt2": get_gpt2_embeddings,
        "clip_text": get_clip_text_embeddings
    }

    for name, func in text_pipeline.items():
        metrics = execute_and_save("text", name, func, ALIGNED_CAPTIONS, device)
        all_metrics.append({**metrics, "Dataset": ds_name})

    send_ntfy_notification(f"Completed {ds_name}", "TFE Pipeline Success")

# --- Save Global Results ---
df_final = pd.DataFrame(all_metrics)
df_final.to_pickle(os.path.join(RESULTS_DIR, "global_unimodal_metrics.pkl"))
print("All datasets and models processed successfully.")


VISION: Flickr8k

Processing VISION with model: resnet50


Indexing resnet50:   0%|          | 0/253 [00:00<?, ?it/s]

Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_vision_resnet50_raw_Flickr8k.npy

Processing VISION with model: mobilenet_v3


Indexing mobilenet_v3:   0%|          | 0/253 [00:00<?, ?it/s]

Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_vision_mobilenet_v3_raw_Flickr8k.npy

Processing VISION with model: vit


...

Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_text_gpt2_raw_Flickr8k.npy

Processing TEXT with model: clip_text


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting CLIP Text:   0%|          | 0/1563 [00:00<?, ?it/s]

Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_text_clip_text_raw_Flickr8k.npy
All datasets and models processed successfully.


In [110]:
for ds in ALL_DATASETS:
    print(f"\nProcessing DeiT for {ds}...")
    df_patch = pd.read_pickle(os.path.join(DATASETS_DIR, f"df_{ds}.pkl"))
    patch_paths = df_patch['image_path'].tolist()
    execute_and_save("vision", "deit", lambda p, d: get_deit_embeddings(p, d, ds), patch_paths, device)

# Final Verification
print("DeiT files in Flickr8k:", len([f for f in os.listdir(os.path.join(DATA_DIR, 'SaliencyMaps_Flickr8k')) if 'deit' in f]))


Processing DeiT for Flickr8k...

Processing VISION with model: deit


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

DeiTModel LOAD REPORT from: facebook/deit-base-distilled-patch16-224
Key                            | Status     | 
-------------------------------+------------+-
distillation_classifier.bias   | UNEXPECTED | 
cls_classifier.weight          | UNEXPECTED | 
distillation_classifier.weight | UNEXPECTED | 
cls_classifier.bias            | UNEXPECTED | 
pooler.dense.weight            | MISSING    | 
pooler.dense.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Targeting Layer: DeiTLayer


Extraction: deit:   0%|          | 0/253 [00:00<?, ?it/s]

...

DeiTModel LOAD REPORT from: facebook/deit-base-distilled-patch16-224
Key                            | Status     | 
-------------------------------+------------+-
distillation_classifier.bias   | UNEXPECTED | 
cls_classifier.weight          | UNEXPECTED | 
distillation_classifier.weight | UNEXPECTED | 
cls_classifier.bias            | UNEXPECTED | 
pooler.dense.weight            | MISSING    | 
pooler.dense.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Targeting Layer: DeiTLayer


Extraction: deit:   0%|          | 0/1563 [00:00<?, ?it/s]

The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). A

Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_vision_deit_raw_Flickr8k.npy
DeiT files in Flickr8k: 0


In [114]:
def extract_vision_features(model, model_name, dataloader, device, target_layer, paths, dataset_name):
    model.eval()
    features, activations = [], {}
    img_idx = 0 # Local counter for this specific run

    def hook_fn(module, input, output):
        # DeiT specific: output is (tensor, weights)
        if isinstance(output, tuple):
            activations['value'] = output[0].detach()
        else:
            activations['value'] = output.detach()

    hook = target_layer.register_forward_hook(hook_fn)
    safe_name = model_name.lower()

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Patching {safe_name}"):
            batch = batch.to(device)
            outputs = model(batch)
            
            # Feature extraction (CLS token)
            if hasattr(outputs, 'last_hidden_state'):
                feat = outputs.last_hidden_state[:, 0, :]
            else:
                feat = outputs[0][:, 0, :]
            features.append(feat.cpu().numpy())

            # --- CRITICAL SAVE SECTION ---
            batch_acts = activations.get('value')
            if batch_acts is not None:
                for i in range(batch_acts.size(0)):
                    # We call the save function here
                    save_saliency_and_overlay(safe_name, batch_acts[i:i+1], img_idx, paths[img_idx], dataset_name)
                    img_idx += 1
                activations['value'] = None 
            else:
                print(f"FAILED to capture activations for {safe_name}")

    hook.remove()
    return np.vstack(features)

In [115]:
# Force clear the pipeline to avoid old references
if 'vision_pipeline' in locals(): del vision_pipeline

for ds in ALL_DATASETS:
    print(f"\n>>> FORCING DeiT PATCH FOR: {ds}")
    
    # Load paths specifically for this dataset
    df_temp = pd.read_pickle(os.path.join(DATASETS_DIR, f"df_{ds}.pkl"))
    current_paths = df_temp['image_path'].tolist()
    
    # Execute specifically for deit
    # We use a wrapper to ensure 'ds' is passed correctly into the extractor
    def deit_wrapper(p, d):
        return get_deit_embeddings(p, d, ds)
        
    execute_and_save("vision", "deit", deit_wrapper, current_paths, device)

    # Verification
    s_path = os.path.join(DATA_DIR, f"SaliencyMaps_{ds}")
    count = len([f for f in os.listdir(s_path) if "deit" in f])
    print(f"RESULT: Found {count} DeiT maps in {ds}")


>>> FORCING DeiT PATCH FOR: Flickr8k

Processing VISION with model: deit


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

DeiTModel LOAD REPORT from: facebook/deit-base-distilled-patch16-224
Key                            | Status     | 
-------------------------------+------------+-
distillation_classifier.bias   | UNEXPECTED | 
cls_classifier.weight          | UNEXPECTED | 
distillation_classifier.weight | UNEXPECTED | 
cls_classifier.bias            | UNEXPECTED | 
pooler.dense.weight            | MISSING    | 
pooler.dense.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Targeting Layer: DeiTLayer


Patching deit:   0%|          | 0/253 [00:00<?, ?it/s]

...

DeiTModel LOAD REPORT from: facebook/deit-base-distilled-patch16-224
Key                            | Status     | 
-------------------------------+------------+-
distillation_classifier.bias   | UNEXPECTED | 
cls_classifier.weight          | UNEXPECTED | 
distillation_classifier.weight | UNEXPECTED | 
cls_classifier.bias            | UNEXPECTED | 
pooler.dense.weight            | MISSING    | 
pooler.dense.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Targeting Layer: DeiTLayer


Patching deit:   0%|          | 0/1563 [00:00<?, ?it/s]

The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). A

Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_vision_deit_raw_Flickr8k.npy
RESULT: Found 0 DeiT maps in ConceptualCaptions


In [117]:
import os
print(os.listdir("/home/aysel/tfe/TFE_Data/SaliencyMaps_Flickr8k")[:10])

['Saliency_clip_vision_3738.npy', 'Saliency_vit_2508.npy', 'Saliency_vit_1941.npy', 'Saliency_clip_vision_7335.npy', 'Saliency_clip_vision_766.npy', 'Saliency_resnet50_2835.npy', 'Saliency_clip_vision_6846.npy', 'Saliency_vit_1589.npy', 'Saliency_vit_4578.npy', 'Saliency_clip_vision_1971.npy']
